# POS Daily CSV

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType, LongType
)
from datetime import datetime

# ── Path configuration ──────────────────────────────────────────────────────
RAW       = "/Volumes/mini_project_team_a3t7/default/mini_project/raw"
BRONZE    = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze"
CKPT_BASE = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze"

# Convenience sub-paths
def raw(f):    return f"{RAW}/{f}"
def bronze(f): return f"{BRONZE}/{f}"
def ckpt(f):   return f"{CKPT_BASE}/{f}"

# Standard metadata columns added to every Bronze table
INGESTION_TS = F.current_timestamp().alias("_ingestion_timestamp")
BATCH_DATE   = F.current_date().alias("_batch_date")

print("Paths configured.")
print(f"  RAW    : {RAW}")
print(f"  BRONZE : {BRONZE}")
print(f"  CKPT   : {CKPT_BASE}")

In [0]:
# 1. Define the Schema
pos_csv_schema = StructType([
    StructField("transaction_id",  StringType(),  False),
    StructField("store_id",        StringType(),  True),
    StructField("product_id",      StringType(),  True),
    StructField("customer_id",     StringType(),  True),
    StructField("timestamp",       StringType(),  True),
    StructField("quantity",        StringType(),  True),
    StructField("unit_price",      StringType(),  True),
    StructField("total_amount",    StringType(),  True),
    StructField("payment_method",  StringType(),  True),
    StructField("channel",         StringType(),  True),
])

# 2. (Auto Loader)
# Point to the DIRECTORY, not a single file, to allow daily files to land there
df_pos_csv_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .schema(pos_csv_schema)
         .option("header", "true")
         .option("nullValue", "")
         .load("/Volumes/mini_project_team_a3t7/default/mini_project/raw/pos_transactions_batch/")
         
         # Convert event time column
         .withColumn("event_time", F.to_timestamp("timestamp"))
         
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date", F.current_date())
         .withColumn("_source", F.lit("pos_batch_csv"))
         
         # Add Watermark
         .withWatermark("event_time", "10 minutes")
)




In [0]:
# 3. Write with Trigger and Checkpointing
(
    df_pos_csv_stream.writeStream
        .format("delta")
        .outputMode("append")                # Use append for incremental daily loads
        .option("checkpointLocation", ckpt("pos_transactions_batch")) 
        .trigger(availableNow=True)          
        .partitionBy("_batch_date")
        .start(bronze("pos_transactions_batch")) 
)

print("Incremental daily load initiated: bronze/pos_transactions_batch")

# Purchase Order

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Schema Definition
po_schema = StructType([
    StructField("po_id",                   StringType(),  False),
    StructField("supplier_id",             StringType(),  True),
    StructField("product_id",              StringType(),  True),
    StructField("order_date",              StringType(),  True),
    StructField("expected_delivery_date",  StringType(),  True),
    StructField("actual_delivery_date",    StringType(),  True),
    StructField("quantity_ordered",        StringType(),  True),
    StructField("unit_cost",               StringType(),  True),
    StructField("status",                  StringType(),  True),
    StructField("quality_rating",          StringType(),  True),
    StructField("delivery_notes",          StringType(),  True),
])

# 2. ReadStream with Auto Loader
df_po_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .schema(po_schema)
         .option("header", "true")
         .option("quote", '"')
         .option("escape", '"')
         .option("multiLine", "true")
         .option("nullValue", "")
         .load(raw("purchase_orders/"))
         
         # Convert event time column
         .withColumn("event_time", F.to_timestamp("order_date"))
         
         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date", F.current_date())
         .withColumn("_source", F.lit("purchase_orders_cdc"))
         .withColumn("_quality_rating_missing", F.col("quality_rating").isNull())
         
         # Add Watermark
         .withWatermark("event_time", "10 minutes")
)

# 3. WriteStream
(
    df_po_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", ckpt("purchase_orders"))
        .trigger(availableNow=True)
        .partitionBy("_batch_date")
        .start(bronze("purchase_orders/"))
)

print("Daily CDC Ingestion initiated: bronze/purchase_orders/")

# Customer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Schema Definition
customer_schema = StructType([
    StructField("customer_id",          StringType(), False),
    StructField("age_group",            StringType(), True),
    StructField("gender",               StringType(), True),
    StructField("zip_code",             StringType(), True),
    StructField("loyalty_tier",         StringType(), True),
    StructField("first_purchase_date",  StringType(), True),
    StructField("total_spend",          StringType(), True),
    StructField("preferred_categories", StringType(), True),
])

# 2. Read Stream using Auto Loader
df_customers_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .schema(customer_schema)
         .option("header", "true")
         .option("nullValue", "")
         .option("quote", '"')
         .option("escape", '"')
         .load(raw("customers/"))

         # Convert event time column
         .withColumn("event_time", F.to_timestamp("first_purchase_date"))

         .withColumn("_ingestion_timestamp", F.current_timestamp())
         .withColumn("_batch_date", F.current_date())
         .withColumn("_source", F.lit("customers_csv_stream"))

         # Add Watermark
         .withWatermark("event_time", "10 minutes")
)

# 3. Write Stream
(
    df_customers_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", ckpt("customers"))
        .trigger(availableNow=True)
        .partitionBy("_batch_date")
        .start(bronze("customers"))
)

print(f"Incremental streaming write started for: {bronze('customers')}")

# External API - Holidays

In [0]:
dbutils.fs.rm(
"/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/external_holiday",
True
)

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime

# 1. Configuration
HOLIDAY_API_KEY = "188da3df024d4526b9060005e0433190"
COUNTRY = "IN"
BRONZE_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/external_holiday/"
CHECKPOINT_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/external_holiday"

# 2. API UDF
def fetch_holiday(trigger_col):
    now = datetime.now()
    url = f"https://holidays.abstractapi.com/v1/?api_key={HOLIDAY_API_KEY}&country={COUNTRY}&year={now.year}&month={now.month}&day={now.day}"
    
    try:
        response = requests.get(url, timeout=10)
        return response.text if response.status_code == 200 else None
    except:
        return None

holiday_udf = F.udf(fetch_holiday, StringType())

# 3. ReadStream (Trigger Source)
df_source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1)
    .option("numPartitions", 1)      # 🔧 CHANGED (Fix partition mismatch issue)
    .load()
)

# 4. Transform
df_bronze = (
    df_source
    .withColumn("raw_content", holiday_udf(F.col("timestamp")))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_batch_date", F.current_date())
    .withColumn("_source", F.lit("holiday_api"))
    .select("raw_content", "_ingestion_timestamp", "_batch_date", "_source")
)

# 5. WriteStream
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .partitionBy("_batch_date")
    .trigger(availableNow=True)
    .start(BRONZE_PATH)
)

query.awaitTermination()

print(f"Holiday Bronze Streaming (Read/Write) for {COUNTRY} finished.")